In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="04-youtube-video-search",
    notebook_name="01_video_search_system_design.ipynb"
)

# YouTube Video Search: System Design

## The Big Idea (For a 12-Year-Old)

Imagine you walk into the **world's biggest library**, except instead of books, there are **one billion videos**. You tell the librarian: "I want funny cat videos." The librarian has to:

1. **Understand what you mean** -- not just the words, but what you *actually* want. Are you looking for compilations? Short clips? Livestreams of cats?
2. **Search through a billion videos** -- but not one by one (that would take years). They need shortcuts.
3. **Understand what each video is about** -- some have great titles ("Hilarious Cat Falls Off Table"), but others have terrible titles ("VID_20240301.mp4"). The librarian needs to actually *look at* the video content.
4. **Rank the results** -- put the best ones first, not just any match.
5. **Do all of this in under 200 milliseconds** -- faster than you can blink.

That is what YouTube's search system does. And in this notebook, we will design it from scratch.

---

## Staff-Level Technical Summary

We design a **YouTube-scale video search system** using:
- **Dual-path architecture**: visual search (embedding-based) + text search (inverted index)
- **Contrastive learning** to train text encoder (BERT) + video encoder (ViT) in a shared embedding space
- **ANN (Approximate Nearest Neighbor)** for sub-millisecond retrieval from billions of vectors
- **Fusing layer** to combine visual + text search results
- **Offline**: MRR, Recall@K | **Online**: total watch time, CTR
- **Scale**: 1B videos, millions of QPS, < 200ms latency

## Step 1: Clarify the Requirements

In any ML system design interview, **always** start by asking clarifying questions. Do not jump into architecture. Think of it like a doctor asking about symptoms before prescribing medicine.

In [ ]:
# Organizing the clarifying requirements as structured data
# This is how you'd think about it systematically in an interview

requirements = {
    "input_type": "Text queries only (no image or video search)",
    "content_type": "Videos only (not images or audio)",
    "relevance_signals": "Visual content + textual metadata (title, description, tags)",
    "training_data": "10 million annotated (video, text query) pairs",
    "language": "English only (for simplicity)",
    "scale": {
        "total_videos": "1 billion",
        "daily_searches": "Hundreds of millions",
        "latency_budget": "< 200 ms end-to-end"
    },
    "personalization": "No -- unlike recommendations, search focuses on query relevance",
}

print("=== YouTube Video Search: Requirements ===")
print(f"\nInput: {requirements['input_type']}")
print(f"Content: {requirements['content_type']}")
print(f"Relevance: {requirements['relevance_signals']}")
print(f"Training data: {requirements['training_data']}")
print(f"Scale: {requirements['scale']['total_videos']} videos")
print(f"Latency: {requirements['scale']['latency_budget']}")
print(f"\nKey insight: Search is NOT personalization.")
print(f"When you search 'Python tutorial', the results should be the same")
print(f"regardless of who you are (unlike your YouTube home feed).")

## Step 2: Frame It as an ML Problem

### The Map Analogy

Imagine a giant map where every video and every search query is a **dot**. If a video is relevant to a query, their dots should be **close together**. If not, they should be **far apart**.

This is called **representation learning** -- we are learning to place things on this map (technically called an "embedding space") so that similar things are near each other.

- **ML Objective**: Rank videos by relevance to the text query
- **Input**: A text query (e.g., "dogs playing indoors")
- **Output**: A ranked list of videos, sorted by relevance
- **ML Category**: Representation learning + ranking

### Why Not Just Use Text Matching?

Because many videos have **terrible metadata**. A video titled "VID_20240301.mp4" that shows dogs playing would be invisible to text search. We need to actually *understand* the video content visually.

## Step 3: High-Level Architecture

The system has **two search pathways** that work together, like having two librarians -- one who reads the book covers (text search) and one who actually watches the videos (visual search).

```
                    Text Query: "dogs playing indoor"
                              |
              +---------------+---------------+
              |                               |
        Visual Search                    Text Search
        (Embedding-based)            (Inverted Index / ES)
              |                               |
      Top-K similar videos          Videos with matching
      by visual content             titles, tags, descriptions
              |                               |
              +---------------+---------------+
                              |
                        Fusing Layer
                   (weighted score combination)
                              |
                      Re-ranking Service
                   (business logic, policies)
                              |
                    Final Ranked Results
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Visualize the dual-path architecture as a flowchart
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('YouTube Video Search: Dual-Path Architecture', fontsize=16, fontweight='bold', pad=20)

# Helper function for boxes
def draw_box(ax, x, y, w, h, text, color='#E8F4FD', edge='#2196F3', fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor=edge, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Query box
draw_box(ax, 4.5, 9, 5, 0.7, 'User Query: "dogs playing indoor"', '#FFF3E0', '#FF9800', 11)

# Visual search path (left)
draw_box(ax, 1, 7.5, 4, 0.7, 'Text Encoder (BERT)\nQuery -> Embedding', '#E3F2FD', '#1565C0')
draw_box(ax, 1, 6, 4, 0.7, 'ANN Search (FAISS/HNSW)\n1B video embeddings', '#E3F2FD', '#1565C0')
draw_box(ax, 1, 4.5, 4, 0.7, 'Top-K Visual Matches\n(content-based)', '#E3F2FD', '#1565C0')

# Text search path (right)
draw_box(ax, 9, 7.5, 4, 0.7, 'Query Processing\nNormalization + Tokenization', '#E8F5E9', '#2E7D32')
draw_box(ax, 9, 6, 4, 0.7, 'Elasticsearch\nInverted Index on metadata', '#E8F5E9', '#2E7D32')
draw_box(ax, 9, 4.5, 4, 0.7, 'Top-K Text Matches\n(title, desc, tags)', '#E8F5E9', '#2E7D32')

# Fusion
draw_box(ax, 4.5, 3, 5, 0.7, 'Fusing Layer\nWeighted Score Combination', '#F3E5F5', '#7B1FA2', 10)

# Re-ranking
draw_box(ax, 4.5, 1.5, 5, 0.7, 'Re-ranking Service\nFreshness, Diversity, Policy', '#FBE9E7', '#BF360C', 10)

# Final results
draw_box(ax, 4.5, 0.2, 5, 0.7, 'Final Ranked Results', '#FFF3E0', '#FF9800', 11)

# Arrows: query to both paths
draw_arrow(ax, 7, 9, 3, 8.2)
draw_arrow(ax, 7, 9, 11, 8.2)

# Arrows: visual path
draw_arrow(ax, 3, 7.5, 3, 6.7)
draw_arrow(ax, 3, 6, 3, 5.2)

# Arrows: text path
draw_arrow(ax, 11, 7.5, 11, 6.7)
draw_arrow(ax, 11, 6, 11, 5.2)

# Arrows to fusion
draw_arrow(ax, 3, 4.5, 7, 3.7)
draw_arrow(ax, 11, 4.5, 7, 3.7)

# Arrow: fusion to reranking
draw_arrow(ax, 7, 3, 7, 2.2)

# Arrow: reranking to final
draw_arrow(ax, 7, 1.5, 7, 0.9)

# Labels for paths
ax.text(3, 8.6, 'Visual Search Path', ha='center', fontsize=11, color='#1565C0', fontstyle='italic')
ax.text(11, 8.6, 'Text Search Path', ha='center', fontsize=11, color='#2E7D32', fontstyle='italic')

plt.tight_layout()
plt.show()

## Step 4: Data Preparation

### Training Data

We have 10 million (video, text query) pairs. Think of these as flash cards: on one side is the search query, on the other side is a video that matches.

Where does this data come from?
- **Click logs**: When a user searches "cats jumping" and clicks on a video, that becomes a positive pair
- **Watch time**: If the user watches most of the video after clicking, it is a *strong* positive signal
- **Human labels**: Annotators can rate (query, video) relevance (expensive but high quality)

In [ ]:
import pandas as pd
import numpy as np

# Simulate the training dataset
np.random.seed(42)

# Synthetic (query, video) pairs
training_data = pd.DataFrame({
    'video_id': ['v_76134', 'v_92167', 'v_02867', 'v_28543', 'v_70310',
                 'v_11111', 'v_22222', 'v_33333', 'v_44444', 'v_55555'],
    'query': [
        'kids swimming in a pool',
        'celebrating graduation ceremony',
        'teenagers playing soccer outdoors',
        'how tensorboard works tutorial',
        'road trip in winter snow driving',
        'funny cat compilation',
        'guitar tutorial for beginners',
        'cooking pasta from scratch',
        'machine learning explained simply',
        'NBA highlights best dunks'
    ],
    'video_title': [
        'Summer Pool Day with the Kids!',
        'Class of 2024 Graduation',
        'Weekend Soccer Match',
        'TensorBoard Complete Guide',
        'Driving Through Colorado Snow',
        'Cats Being Cats - Try Not To Laugh',
        'Learn Guitar in 30 Days',
        'Italian Pasta Masterclass',
        'ML Crash Course',
        'Top 10 Dunks of the Season'
    ],
    'split': ['train', 'train', 'train', 'train', 'train',
              'val', 'val', 'val', 'test', 'test']
})

print("=== Sample Training Data ===")
print("Each row is a (query, video) pair where the video IS relevant to the query.")
print()
print(training_data.to_string(index=False))

print(f"\n=== Data Split ===")
for split in ['train', 'val', 'test']:
    count = (training_data['split'] == split).sum()
    print(f"  {split}: {count} samples")

print("\nIn reality: 10M pairs total, with 80/10/10 split")
print("Sources: click logs (implicit) + human annotation (explicit)")

## Step 5: Feature Engineering

### Text Features

Before we can feed text into an ML model, we need to clean it up and convert it to numbers. Think of it like translating English into a language that computers understand.

The pipeline: **Raw Text --> Normalize --> Tokenize --> Token IDs**

### Video Features

Videos are even harder. A single video might contain:
- **Visual frames**: What you see (objects, scenes, actions)
- **Audio**: Music, speech, sound effects
- **Metadata**: Title, description, tags, captions

We sample frames from the video, feed them through a vision model (like ViT), and aggregate the frame embeddings into a single video embedding.

In [ ]:
import re
import unicodedata

# === Text Preprocessing Pipeline ===

def normalize_text(text):
    """
    Clean up text so that 'DOG!', 'dogs', and 'Dog' all become 'dog'.
    
    12-year-old version: It is like a spell checker that also makes
    everything lowercase and removes weird symbols.
    """
    # Lowercase
    text = text.lower()
    # NFKD normalization (decompose combined characters)
    text = unicodedata.normalize('NFKD', text)
    # Strip accents
    text = ''.join(c for c in text if not unicodedata.combining(c))
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def simple_tokenize(text):
    """Simple word-level tokenization (split on spaces)."""
    return text.split()


def tokens_to_ids(tokens, vocab):
    """
    Convert tokens to integer IDs using a vocabulary lookup table.
    Unknown tokens get a special [UNK] id.
    """
    unk_id = vocab.get('[UNK]', 0)
    return [vocab.get(t, unk_id) for t in tokens]


# Build a simple vocabulary from our sample data
all_queries = training_data['query'].tolist()
all_words = set()
for q in all_queries:
    all_words.update(simple_tokenize(normalize_text(q)))

vocab = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3}
for i, word in enumerate(sorted(all_words), start=4):
    vocab[word] = i

# Demo the pipeline
demo_queries = [
    "Funny CAT Videos!!!",
    "how TensorBoard works??",
    "NBA  highlights   BEST dunks"
]

print("=== Text Preprocessing Pipeline ===")
for query in demo_queries:
    normalized = normalize_text(query)
    tokens = simple_tokenize(normalized)
    ids = tokens_to_ids(tokens, vocab)
    print(f"\n  Raw:        '{query}'")
    print(f"  Normalized: '{normalized}'")
    print(f"  Tokens:     {tokens}")
    print(f"  Token IDs:  {ids}")

print(f"\nVocabulary size: {len(vocab)} tokens")
print(f"\nIn production: BERT uses WordPiece tokenization (subword level)")
print(f"  'unforgettable' -> ['un', '##for', '##get', '##table']")
print(f"  This handles unseen words gracefully.")

In [ ]:
# === Video Preprocessing Pipeline (Simulated) ===

def simulate_video_preprocessing(video_id, n_frames=8, frame_size=(224, 224, 3)):
    """
    Simulate the video preprocessing pipeline.
    
    Real pipeline:
    1. Decode video file
    2. Sample N frames (uniformly or at keyframes)
    3. Resize each frame to 224x224
    4. Normalize pixel values to [0, 1]
    5. Feed through ViT -> get frame embeddings
    6. Aggregate (average pool) -> single video embedding
    
    12-year-old version:
    Imagine flipping through a photo album of the video.
    You pick 8 photos evenly spaced, resize them to the same size,
    show each to a smart 'art critic' (ViT), and average their opinions.
    """
    np.random.seed(hash(video_id) % 2**31)
    
    # Step 1-3: Sample and resize frames (simulated as random images)
    frames = np.random.rand(n_frames, *frame_size)
    
    # Step 4: Normalize (already [0,1] since random)
    
    # Step 5: Simulate ViT encoding (each frame -> 256-dim embedding)
    embedding_dim = 256
    frame_embeddings = np.random.randn(n_frames, embedding_dim)
    # Normalize each frame embedding
    frame_embeddings = frame_embeddings / np.linalg.norm(frame_embeddings, axis=1, keepdims=True)
    
    # Step 6: Average pooling -> single video embedding
    video_embedding = frame_embeddings.mean(axis=0)
    video_embedding = video_embedding / np.linalg.norm(video_embedding)  # L2 normalize
    
    return {
        'video_id': video_id,
        'num_frames': n_frames,
        'frame_shape': frame_size,
        'frame_embeddings_shape': frame_embeddings.shape,
        'video_embedding_shape': video_embedding.shape,
        'video_embedding': video_embedding
    }

# Demo
result = simulate_video_preprocessing('v_76134')

print("=== Video Preprocessing Pipeline ===")
print(f"\n  Video ID: {result['video_id']}")
print(f"  Sampled frames: {result['num_frames']}")
print(f"  Each frame shape: {result['frame_shape']} (H x W x C)")
print(f"  Frame embeddings: {result['frame_embeddings_shape']} (N_frames x embedding_dim)")
print(f"  Final video embedding: {result['video_embedding_shape']} (single vector)")
print(f"  Embedding norm: {np.linalg.norm(result['video_embedding']):.4f} (unit normalized)")
print(f"\n  First 10 values: {result['video_embedding'][:10].round(4)}")

print("\n=== Why Sample Frames Instead of Using All? ===")
print("  A 5-minute video at 30fps = 9,000 frames")
print("  Processing all frames: ~30 seconds on a GPU")
print("  Processing 8 sampled frames: ~0.1 seconds")
print("  Quality loss: minimal for search (adjacent frames are nearly identical)")

## Step 6: Model Development -- Contrastive Learning

### The Matching Game Analogy

Imagine you are playing a matching card game. You have two decks:
- **Deck A**: Search queries ("funny cat videos", "guitar tutorial", etc.)
- **Deck B**: Videos (cat compilation, guitar lesson, cooking show, etc.)

The game is: for each query card, find the matching video card.

**Contrastive learning** trains the model to play this game. In each round (batch):
- **Positive pairs**: The query and its matching video should have **high similarity** (close on the map)
- **Negative pairs**: The query and all NON-matching videos should have **low similarity** (far on the map)

The more negative examples you show, the better the model learns to discriminate. This is why **large batch sizes** are so important in contrastive learning.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === Dual Encoder Architecture ===

class TextEncoder(nn.Module):
    """
    Encodes a text query into a dense embedding.
    
    12-year-old version:
    This is the 'text brain' that reads a search query and
    summarizes it as a list of numbers (embedding).
    
    In production: This would be BERT or a Sentence Transformer.
    Here we simulate with a simple embedding + MLP.
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, output_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, token_ids):
        # token_ids: (batch_size, seq_len)
        embeds = self.embedding(token_ids)       # (batch, seq_len, embed_dim)
        pooled = embeds.mean(dim=1)               # (batch, embed_dim) -- average pooling
        output = self.encoder(pooled)             # (batch, output_dim)
        output = F.normalize(output, p=2, dim=1)  # L2 normalize
        return output


class VideoEncoder(nn.Module):
    """
    Encodes video frames into a dense embedding.
    
    12-year-old version:
    This is the 'video brain' that watches sampled frames and
    summarizes the video as a list of numbers.
    
    In production: This would be ViT applied to each frame, then pooled.
    Here we simulate with a simple CNN + MLP.
    """
    def __init__(self, frame_feature_dim=512, hidden_dim=256, output_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(frame_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, frame_features):
        # frame_features: (batch_size, n_frames, feature_dim)
        pooled = frame_features.mean(dim=1)        # (batch, feature_dim) -- average pooling
        output = self.encoder(pooled)              # (batch, output_dim)
        output = F.normalize(output, p=2, dim=1)   # L2 normalize
        return output


class DualEncoder(nn.Module):
    """
    The complete dual-encoder model.
    Text encoder + Video encoder trained together with contrastive loss.
    """
    def __init__(self, vocab_size, frame_feature_dim=512, embedding_dim=256):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, output_dim=embedding_dim)
        self.video_encoder = VideoEncoder(frame_feature_dim, output_dim=embedding_dim)
        self.temperature = nn.Parameter(torch.tensor(0.07))  # Learnable temperature
    
    def forward(self, token_ids, frame_features):
        text_emb = self.text_encoder(token_ids)       # (batch, emb_dim)
        video_emb = self.video_encoder(frame_features) # (batch, emb_dim)
        return text_emb, video_emb


# Instantiate
vocab_size = len(vocab)
model = DualEncoder(vocab_size=vocab_size, frame_feature_dim=512, embedding_dim=256)

print("=== Dual Encoder Architecture ===")
print(f"\nText Encoder:")
print(f"  Input: token IDs (batch, seq_len)")
print(f"  Output: text embedding (batch, 256)")
print(f"\nVideo Encoder:")
print(f"  Input: frame features (batch, n_frames, 512)")
print(f"  Output: video embedding (batch, 256)")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nBoth encoders output L2-normalized 256-dim vectors.")
print(f"Similarity = dot product (which equals cosine similarity when L2-normalized).")

In [ ]:
# === Contrastive Loss (InfoNCE / CLIP-style) ===

def contrastive_loss(text_embeddings, video_embeddings, temperature=0.07):
    """
    Symmetric contrastive loss (like CLIP).
    
    12-year-old version:
    In a batch of N (query, video) pairs, each query should match
    its own video (the diagonal) and NOT match any other video.
    
    Think of a classroom with 8 students, each holding a flashcard.
    Student 1's query should match Student 1's video,
    but NOT Student 2's, 3's, ..., 8's videos.
    
    Staff-level detail:
    - Compute NxN similarity matrix (all query-video pairs)
    - Diagonal entries are positives (matching pairs)
    - Off-diagonal entries are negatives (non-matching)
    - Apply softmax + cross-entropy in both directions
    - Temperature controls how 'sharp' the softmax is
    """
    batch_size = text_embeddings.shape[0]
    
    # Compute similarity matrix: (N, N)
    # Since embeddings are L2-normalized, dot product = cosine similarity
    logits = torch.matmul(text_embeddings, video_embeddings.T) / temperature
    
    # Labels: the diagonal (each query matches its own video)
    labels = torch.arange(batch_size)
    
    # Symmetric loss: text->video + video->text
    loss_t2v = F.cross_entropy(logits, labels)        # For each query, find matching video
    loss_v2t = F.cross_entropy(logits.T, labels)      # For each video, find matching query
    
    loss = (loss_t2v + loss_v2t) / 2
    return loss


# Demonstrate with a small batch
batch_size = 4
emb_dim = 256

# Simulate embeddings
torch.manual_seed(42)
text_embs = F.normalize(torch.randn(batch_size, emb_dim), dim=1)
video_embs = F.normalize(torch.randn(batch_size, emb_dim), dim=1)

# Compute loss
loss = contrastive_loss(text_embs, video_embs)

# Show the similarity matrix
sim_matrix = torch.matmul(text_embs, video_embs.T)

print("=== Contrastive Loss Demo ===")
print(f"\nBatch size: {batch_size}")
print(f"\nSimilarity Matrix (before training -- random, so nothing matches):")
print(f"           Video_0  Video_1  Video_2  Video_3")
for i in range(batch_size):
    row = '  '.join(f'{sim_matrix[i, j].item():7.4f}' for j in range(batch_size))
    label = ' <-- should be highest' if True else ''
    print(f"  Query_{i}: {row}")

print(f"\nDiagonal should be highest (positive pairs).")
print(f"Off-diagonal should be lowest (negative pairs).")
print(f"\nContrastive loss: {loss.item():.4f}")
print(f"Random baseline loss (batch_size=4): {np.log(4):.4f}")
print(f"\nAfter training, diagonal values >> off-diagonal values.")

In [ ]:
# === Training Loop ===

def generate_synthetic_batch(batch_size, vocab_size, seq_len=10, n_frames=8, frame_dim=512):
    """
    Generate a synthetic training batch.
    In production, this comes from the real (query, video) dataset.
    """
    token_ids = torch.randint(1, vocab_size, (batch_size, seq_len))
    frame_features = torch.randn(batch_size, n_frames, frame_dim)
    return token_ids, frame_features


# Training setup
model = DualEncoder(vocab_size=vocab_size, frame_feature_dim=512, embedding_dim=256)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 32
n_epochs = 30
n_batches_per_epoch = 20  # Simulated

print("=== Training the Dual Encoder with Contrastive Learning ===")
print(f"Batch size: {batch_size} (each batch gives {batch_size} positives, {batch_size*(batch_size-1)} negatives)")
print(f"Epochs: {n_epochs}")
print(f"Embedding dimension: 256")
print()

losses = []
model.train()

for epoch in range(n_epochs):
    epoch_loss = 0
    for _ in range(n_batches_per_epoch):
        token_ids, frame_features = generate_synthetic_batch(batch_size, vocab_size)
        
        # Forward pass
        text_emb, video_emb = model(token_ids, frame_features)
        
        # Compute contrastive loss
        loss = contrastive_loss(text_emb, video_emb, temperature=model.temperature)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / n_batches_per_epoch
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', linewidth=2)
plt.axhline(y=np.log(batch_size), color='r', linestyle='--', label=f'Random baseline (ln({batch_size})={np.log(batch_size):.2f})')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Contrastive Loss', fontsize=12)
plt.title('Training Loss: Dual Encoder with Contrastive Learning', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal loss: {losses[-1]:.4f}")
print(f"Random baseline: {np.log(batch_size):.4f}")
print(f"\nLoss below random baseline = model is learning to match queries to videos!")

## Step 7: Evaluation Metrics

### Offline Metrics

How do we know if our model is good *before* deploying it to real users?

| Metric | What It Measures | Good for Us? | Why? |
|--------|-----------------|--------------|------|
| **Precision@K** | Fraction of top-K that are relevant | Not ideal | With 1 relevant video per query, precision@10 maxes at 0.1 |
| **Recall@K** | Is the relevant video in top-K? | Decent | But cannot distinguish rank 1 from rank K |
| **MRR** | Average of 1/rank of first relevant result | **Best** | Rewards finding the relevant video earlier |
| **NDCG** | Considers graded relevance + position | Great (if graded labels) | Standard for search with graded relevance |

### Online Metrics

| Metric | What It Measures | Limitation |
|--------|-----------------|------------|
| **CTR** | Click-through rate | Clickbait gets high CTR |
| **Video Completion Rate** | How much of the video is watched | Short videos get unfair advantage |
| **Total Watch Time** | Total time watching search results | **Best proxy** -- users watch more when results are good |

In [ ]:
# === Evaluation Metrics Implementation ===

def recall_at_k(relevant_rank, k):
    """
    Recall@K: Is the relevant video in the top K results?
    Returns 1 if yes, 0 if no.
    
    12-year-old version: 
    'Did my video make it into the top K list?'
    """
    return 1.0 if relevant_rank <= k else 0.0


def reciprocal_rank(relevant_rank):
    """
    Reciprocal Rank: 1/position of the first relevant result.
    
    12-year-old version:
    If your video is ranked 1st: score = 1.0 (perfect!)
    If ranked 2nd: score = 0.5
    If ranked 10th: score = 0.1
    If ranked 100th: score = 0.01 (terrible)
    """
    return 1.0 / relevant_rank


def mean_reciprocal_rank(relevant_ranks):
    """
    MRR: Average reciprocal rank across all queries.
    
    Formula: MRR = (1/m) * sum(1/rank_i) for all queries
    """
    return np.mean([reciprocal_rank(r) for r in relevant_ranks])


def ndcg_at_k(relevance_scores, k):
    """
    NDCG@K: Normalized Discounted Cumulative Gain.
    
    Works with graded relevance (not just binary).
    DCG = sum(relevance_i / log2(i+1)) for positions 1..k
    NDCG = DCG / Ideal_DCG (perfect ranking)
    """
    def dcg(scores, k):
        scores = np.array(scores[:k])
        positions = np.arange(1, len(scores) + 1)
        return np.sum(scores / np.log2(positions + 1))
    
    actual_dcg = dcg(relevance_scores, k)
    ideal_dcg = dcg(sorted(relevance_scores, reverse=True), k)
    
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0


# === Demo with examples ===

# Simulate: for 6 queries, at what rank was the relevant video found?
query_results = [
    {"query": "funny cat videos", "relevant_rank": 1},
    {"query": "guitar tutorial", "relevant_rank": 3},
    {"query": "NBA highlights", "relevant_rank": 2},
    {"query": "cooking pasta", "relevant_rank": 5},
    {"query": "ML crash course", "relevant_rank": 1},
    {"query": "road trip vlog", "relevant_rank": 8},
]

ranks = [q['relevant_rank'] for q in query_results]
mrr = mean_reciprocal_rank(ranks)

print("=== Evaluation Metrics Demo ===")
print(f"\n{'Query':<25} {'Rank':>5}  {'RR':>6}  {'Recall@5':>9}  {'Recall@10':>10}")
print("-" * 65)
for q in query_results:
    rr = reciprocal_rank(q['relevant_rank'])
    r5 = recall_at_k(q['relevant_rank'], 5)
    r10 = recall_at_k(q['relevant_rank'], 10)
    print(f"  {q['query']:<23} {q['relevant_rank']:>5}  {rr:>6.3f}  {r5:>9.1f}  {r10:>10.1f}")

print(f"\n{'MRR':>47}: {mrr:.4f}")
print(f"{'Recall@5':>47}: {np.mean([recall_at_k(r, 5) for r in ranks]):.4f}")
print(f"{'Recall@10':>47}: {np.mean([recall_at_k(r, 10) for r in ranks]):.4f}")

# NDCG example with graded relevance
print(f"\n=== NDCG Example (Graded Relevance) ===")
graded = [3, 0, 2, 0, 1, 0, 0, 1, 0, 0]  # Relevance scores for top-10 results
ndcg5 = ndcg_at_k(graded, 5)
ndcg10 = ndcg_at_k(graded, 10)
print(f"  Top-10 relevance scores: {graded}")
print(f"  NDCG@5:  {ndcg5:.4f}")
print(f"  NDCG@10: {ndcg10:.4f}")

In [ ]:
# === Visualize How MRR Works ===

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scenarios = [
    {"name": "Great Model", "ranks": [1, 1, 2, 1, 3, 1], "color": "#4CAF50"},
    {"name": "OK Model", "ranks": [1, 3, 2, 5, 1, 8], "color": "#FF9800"},
    {"name": "Bad Model", "ranks": [10, 25, 50, 8, 30, 15], "color": "#F44336"},
]

for ax, scenario in zip(axes, scenarios):
    ranks = scenario['ranks']
    rr_values = [1.0/r for r in ranks]
    mrr_val = np.mean(rr_values)
    
    bars = ax.bar(range(len(ranks)), rr_values, color=scenario['color'], alpha=0.7, edgecolor='black')
    ax.axhline(y=mrr_val, color='red', linestyle='--', linewidth=2, label=f'MRR = {mrr_val:.3f}')
    ax.set_xlabel('Query Index', fontsize=11)
    ax.set_ylabel('Reciprocal Rank (1/rank)', fontsize=11)
    ax.set_title(f'{scenario["name"]}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add rank labels on bars
    for bar, rank in zip(bars, ranks):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'rank={rank}', ha='center', va='bottom', fontsize=8)

plt.suptitle('MRR: Mean Reciprocal Rank Across Different Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: MRR rewards finding the relevant video EARLY.")
print("A model that consistently ranks relevant videos in top 3 gets MRR close to 1.0.")
print("A model that buries relevant videos beyond rank 10 gets MRR close to 0.")

## Step 8: Serving Architecture

### Three Pipelines

The full system has three pipelines that work together:

**1. Prediction Pipeline (Online -- when a user searches)**
```
User Query -> Text Encoder -> query embedding
                   |
                   +--> ANN Service --> top-K visual matches
                   +--> Elasticsearch --> text-matched videos
                   |
                   v
              Fusing Layer --> combined ranked list
                   |
              Re-ranking --> apply business rules
                   |
              Display to User
```

**2. Video Indexing Pipeline (Offline -- when a new video is uploaded)**
```
New Video --> Video Encoder --> video embedding --> ANN Index
```

**3. Text Indexing Pipeline (Offline -- when a new video is uploaded)**
```
New Video --> Extract metadata (title, description, tags)
         --> Auto-tagger (generate tags from content)
         --> Elasticsearch Index
```

The key insight: **video embeddings are computed OFFLINE** (when uploaded), but **query embeddings are computed ONLINE** (at search time). This is critical for latency -- we cannot encode a video in real-time.

In [ ]:
# === Simulating the Complete Serving Pipeline ===

class VideoSearchSystem:
    """
    End-to-end video search system simulation.
    
    12-year-old version:
    The system has two librarians working together:
    - Librarian A (visual): 'I watched all the videos and remember what they look like'
    - Librarian B (text): 'I read all the titles and descriptions'
    When you search, both librarians find their best matches,
    then a manager combines their lists and picks the final top results.
    """
    def __init__(self, n_videos=1000, embedding_dim=256):
        self.n_videos = n_videos
        self.embedding_dim = embedding_dim
        
        # Simulate video database
        np.random.seed(42)
        categories = ['music', 'sports', 'cooking', 'tech', 'comedy', 'education', 'travel', 'gaming']
        self.videos = []
        for i in range(n_videos):
            cat = categories[i % len(categories)]
            self.videos.append({
                'id': f'v_{i:05d}',
                'title': f'{cat.title()} Video #{i}',
                'category': cat,
                'views': np.random.randint(100, 1000000),
                'upload_days_ago': np.random.randint(1, 365)
            })
        
        # Pre-computed video embeddings (offline indexing)
        self.video_embeddings = np.random.randn(n_videos, embedding_dim)
        self.video_embeddings /= np.linalg.norm(self.video_embeddings, axis=1, keepdims=True)
        
        # Simulate text index (inverted index)
        self.text_index = {}
        for i, video in enumerate(self.videos):
            for word in video['title'].lower().split():
                if word not in self.text_index:
                    self.text_index[word] = []
                self.text_index[word].append(i)
    
    def visual_search(self, query_embedding, top_k=50):
        """ANN-based visual search using dot product similarity."""
        similarities = self.video_embeddings @ query_embedding
        top_indices = np.argsort(-similarities)[:top_k]
        return [(idx, similarities[idx]) for idx in top_indices]
    
    def text_search(self, query_text, top_k=50):
        """Inverted index text search (simulating Elasticsearch)."""
        query_words = query_text.lower().split()
        doc_scores = {}
        for word in query_words:
            if word in self.text_index:
                for doc_idx in self.text_index[word]:
                    doc_scores[doc_idx] = doc_scores.get(doc_idx, 0) + 1
        
        sorted_docs = sorted(doc_scores.items(), key=lambda x: -x[1])[:top_k]
        # Normalize scores to [0, 1]
        max_score = sorted_docs[0][1] if sorted_docs else 1
        return [(idx, score / max_score) for idx, score in sorted_docs]
    
    def fuse_results(self, visual_results, text_results, visual_weight=0.6, text_weight=0.4):
        """
        Combine visual and text search results with weighted scoring.
        
        12-year-old version:
        Visual librarian's opinion counts for 60%.
        Text librarian's opinion counts for 40%.
        This balances content understanding with metadata matching.
        """
        combined = {}
        for idx, score in visual_results:
            combined[idx] = combined.get(idx, 0) + visual_weight * score
        for idx, score in text_results:
            combined[idx] = combined.get(idx, 0) + text_weight * score
        
        return sorted(combined.items(), key=lambda x: -x[1])
    
    def rerank(self, fused_results, top_k=10):
        """
        Apply business rules (freshness boost, diversity).
        """
        reranked = []
        for idx, score in fused_results[:top_k * 2]:  # Consider 2x candidates
            video = self.videos[idx]
            # Freshness boost: newer videos get a small boost
            freshness_boost = max(0, 1 - video['upload_days_ago'] / 365) * 0.1
            # Popularity signal
            popularity_boost = np.log1p(video['views']) / 20 * 0.05
            final_score = score + freshness_boost + popularity_boost
            reranked.append((idx, final_score))
        
        reranked.sort(key=lambda x: -x[1])
        return reranked[:top_k]
    
    def search(self, query_text, top_k=10):
        """Complete search pipeline."""
        # Step 1: Encode query (simulated)
        np.random.seed(hash(query_text) % 2**31)
        query_embedding = np.random.randn(self.embedding_dim)
        query_embedding /= np.linalg.norm(query_embedding)
        
        # Step 2: Dual-path search
        visual_results = self.visual_search(query_embedding, top_k=50)
        text_results = self.text_search(query_text, top_k=50)
        
        # Step 3: Fuse
        fused = self.fuse_results(visual_results, text_results)
        
        # Step 4: Rerank
        final = self.rerank(fused, top_k)
        
        return final, len(visual_results), len(text_results)


# Demo
system = VideoSearchSystem(n_videos=1000)

query = "music concert live"
results, n_visual, n_text = system.search(query, top_k=10)

print(f"=== Search Results for: '{query}' ===")
print(f"\nVisual search returned: {n_visual} candidates")
print(f"Text search returned:   {n_text} candidates")
print(f"\n{'Rank':<6} {'Video ID':<12} {'Title':<30} {'Score':>8} {'Views':>10} {'Days Ago':>10}")
print("-" * 80)
for rank, (idx, score) in enumerate(results, 1):
    v = system.videos[idx]
    print(f"  {rank:<4} {v['id']:<12} {v['title']:<30} {score:>8.4f} {v['views']:>10,} {v['upload_days_ago']:>10}")

## Key Takeaways

1. **Dual-path architecture** -- Visual search (embedding-based) catches videos with bad metadata; text search (inverted index) is fast and reliable for well-tagged content.

2. **Contrastive learning** -- Trains text and video encoders to map queries and videos into the same embedding space. Larger batch sizes give more negatives, which improves learning.

3. **Frame-level over video-level encoding** -- Sampling 8 frames is 100x faster than processing all frames, with minimal quality loss for search relevance.

4. **MRR is the primary offline metric** -- With one relevant video per query, MRR directly rewards finding it earlier. Total watch time is the best online metric.

5. **Offline video indexing is critical** -- Video embeddings are computed when uploaded (offline), while query embeddings are computed at search time (online). This keeps latency under 200ms.

6. **Fusing + re-ranking** -- Weighted score combination merges both search paths, then business rules (freshness, diversity, policy) produce the final ranking.

7. **Auto-tagging improves coverage** -- When uploaders provide bad/no tags, an auto-tagger model generates tags from video content, improving text search recall.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)